# Insturctions to follow before executing this script

## Step 1: Create the Isolated phase2_trellis (Run in Terminal)



### Tunnel for Jupyter
ssh -f -N -L 5402:scn33-mn:8888 rahkus@login.npsf.cdac.in
### Tunnel for Gradio
ssh -f -N -L 7862:scn33-mn:7860 rahkus@login.npsf.cdac.in


In [1]:
# Do not uninstall CUDA runtime packages automatically. This environment may be shared
# with other jobs; verify the installed stack in the setup cell before changing it.
print("CUDA package removal is disabled; runtime package state is preserved.")


In [2]:
# Read-only PyTorch/CUDA compatibility gate; no forced CUDA-library reinstall.
import torch
print(f"PyTorch: {torch.__version__} | Compiled CUDA: {torch.version.cuda} | cuDNN: {torch.backends.cudnn.version()}")
assert torch.__version__.startswith("2.6.0"), f"Expected PyTorch 2.6.0, got {torch.__version__}"
assert torch.version.cuda == "12.4", f"Expected CUDA 12.4 build, got {torch.version.cuda}"


ERROR: Could not find a version that satisfies the requirement nvidia-cudnn-cu12==9.1.0.70 (from versions: none)
ERROR: No matching distribution found for nvidia-cudnn-cu12==9.1.0.70


In [3]:
!python -c "import torch; print(f'PyTorch: {torch.__version__} | Compiled CUDA: {torch.version.cuda} | Active cuDNN: {torch.backends.cudnn.version()}')"

PyTorch: 2.6.0+cu124 | Compiled CUDA: 12.4 | Active cuDNN: 90100


In [4]:
!pip list | grep -iE 'cu13|cuda-toolkit'

In [5]:
!pip list | grep -i cu12

nvidia-cublas-cu12        12.4.5.8
nvidia-cuda-cupti-cu12    12.4.127
nvidia-cuda-nvrtc-cu12    12.4.127
nvidia-cuda-runtime-cu12  12.4.127
nvidia-cudnn-cu12         9.1.0.70
nvidia-cufft-cu12         11.2.1.3
nvidia-curand-cu12        10.3.5.147
nvidia-cusolver-cu12      11.6.1.9
nvidia-cusparse-cu12      12.3.1.170
nvidia-cusparselt-cu12    0.6.2
nvidia-nccl-cu12          2.21.5
nvidia-nvjitlink-cu12     12.4.127
nvidia-nvtx-cu12          12.4.127
torch                     2.6.0+cu124
torchvision               0.21.0+cu124


In [6]:
import os

# ==============================================================================
# GLOBAL CONFIGURATION & TOGGLES
# ==============================================================================
FORCE_RECOMPILE = False    # Set to True only if you need to wipe and rebuild C++ extensions
ENABLE_DISK_PURGE = False  # Set False for safety. Leaves models on SSD for mmap.
TARGET_GPU = "1"           # Change to 0, 1, 2, or 3 for node isolation.

# 1. ISOLATE GPU: MUST BE THE ABSOLUTE FIRST LINE EXECUTED
os.environ["CUDA_VISIBLE_DEVICES"] = TARGET_GPU
print(f"[SYSTEM] Notebook strictly locked to GPU: {os.environ['CUDA_VISIBLE_DEVICES']}")

# 2. FIX GRADIO PROXY CRASH: Force Python to bypass proxy for localhost pings
os.environ["no_proxy"] = "localhost, 127.0.0.1, ::1, 0.0.0.0"
os.environ["NO_PROXY"] = "localhost, 127.0.0.1, ::1, 0.0.0.0"

# 3. THE FROZEN RULE: Core Config
import core_config
print("[SYSTEM] Core proxies, storage mapping, and HF Auth executed automatically.")
from core_config import SYSTEM_DIRS

# 4. DISABLE BENCHMARKING (Prevents cuDNN workspace fragmentation panics)
#import torch
#torch.backends.cudnn.benchmark = False

[SYSTEM] Notebook strictly locked to GPU: 1
[CORE] Secrets & Config loaded successfully from /nlsasfs/home/lmedge/rahkus/.env
[CORE] Cleaned trash at /nlsasfs/home/lmedge/rahkus/.local/share/Trash/files
[CORE] Cleaned trash at /nlsasfs/home/lmedge/rahkus/.local/share/Trash/info
[CORE] Cleaned trash at /nlsasfs/home/lmedge/rahkus/.jupyter/lab/workspaces
[SYSTEM] Core proxies, storage mapping, and HF Auth executed automatically.


In [7]:
# Read-only process inspection; never kill a hard-coded PID.
!ps -u "$USER" -o pid,etime,cmd | grep -E '[g]radio|[p]ipeline_trellis' || true


/bin/bash: line 1: kill: (9) - No such process
/bin/bash: line 1: kill: (2386856) - No such process


In [8]:
!nvidia-smi

Sun Sep 13 00:20:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.07              Driver Version: 550.90.07      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          On  |   00000000:07:00.0 Off |                    0 |
| N/A   31C    P0             50W /  400W |       1MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [9]:
# Verify notebook UI packages without upgrading them during model setup.
import importlib.metadata
for package in ("jupyter", "ipywidgets", "ipykernel"):
    try:
        print(f"{package}=={importlib.metadata.version(package)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package}=MISSING (install only in phase2_3d after explicit approval)")


In [10]:
import os
import sys
import subprocess
import shutil
import importlib.util
from core_config import SYSTEM_DIRS

def run_cmd(cmd, cwd=None):
    subprocess.run(cmd, cwd=cwd, check=True)

def is_installed(pkg_name):
    return importlib.util.find_spec(pkg_name) is not None

# ==============================================================================
# 0. THE NUCLEAR UNINSTALL & CUDA 13 PURGE
# ==============================================================================
print("0. Preserving existing CUDA runtime packages; no automatic CUDA 13 package removal.")

if FORCE_RECOMPILE:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "cumesh", "flex_gemm", "o_voxel", "nvdiffrast", "nvdiffrec_render", "xformers"], check=False)

# ==============================================================================
# 1. IRONCLAD PYTORCH 2.6.0 & COMPATIBLE XFORMERS INSTALLATION
# ==============================================================================
print("1. Enforcing PyTorch 2.6.0 (CUDA 12.4) and compatible xformers...")
run_cmd([sys.executable, "-m", "pip", "install", "torch==2.6.0", "torchvision==0.21.0", "--index-url", "https://download.pytorch.org/whl/cu124"])

# Keep PyTorch's CUDA 12.4 transitive libraries pinned by the PyTorch wheel.
# Do not force-reinstall cuDNN/cuBLAS/NVRTC/NCCL independently.
# xformers 0.0.29.post3 is the release line built for PyTorch 2.6.0.
run_cmd([sys.executable, "-m", "pip", "install", "xformers==0.0.29.post3", "--no-deps"])

# ==============================================================================
# 2. OFFICIAL SETUP.SH DEPENDENCIES
# ==============================================================================
print("2. Installing Basic Dependencies...")
run_cmd([
    sys.executable, "-m", "pip", "install", 
    "torch==2.6.0", "torchvision==0.21.0",
    "imageio", "imageio-ffmpeg", "tqdm", "easydict", "opencv-python-headless", 
    "ninja", "trimesh", "transformers==4.56.1", "gradio==6.0.1", "tensorboard", "pandas", 
    "lpips", "zstandard", "kornia", "timm", "pillow"
])

run_cmd([sys.executable, "-m", "pip", "install", "git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8"])

# ==============================================================================
# 3. C++ EXTENSIONS COMPILATION
# ==============================================================================
extensions_exist = is_installed("nvdiffrast") and is_installed("nvdiffrec_render") and is_installed("cumesh") and is_installed("flex_gemm") and is_installed("o_voxel")

if not FORCE_RECOMPILE and extensions_exist:
    print("3. ✅ 3D C++ Extensions already detected! Skipping heavy compilation.")
else:
    print("3. Compiling C++ Extensions...")
    EXT_DIR = os.path.join(SYSTEM_DIRS['cache'], "trellis2_extensions")
    os.makedirs(EXT_DIR, exist_ok=True)

    nd_dir = os.path.join(EXT_DIR, "nvdiffrast")
    if os.path.exists(nd_dir): shutil.rmtree(nd_dir)
    run_cmd(["git", "clone", "-b", "v0.4.0", "https://github.com/NVlabs/nvdiffrast.git", nd_dir])
    run_cmd([sys.executable, "-m", "pip", "install", "--no-cache-dir", nd_dir, "--no-build-isolation"])

    nr_dir = os.path.join(EXT_DIR, "nvdiffrec")
    if os.path.exists(nr_dir): shutil.rmtree(nr_dir)
    run_cmd(["git", "clone", "-b", "renderutils", "https://github.com/JeffreyXiang/nvdiffrec.git", nr_dir])
    run_cmd([sys.executable, "-m", "pip", "install", "--no-cache-dir", nr_dir, "--no-build-isolation"])

    cm_dir = os.path.join(EXT_DIR, "CuMesh")
    if os.path.exists(cm_dir): shutil.rmtree(cm_dir)
    run_cmd(["git", "clone", "https://github.com/JeffreyXiang/CuMesh.git", cm_dir, "--recursive"])
    run_cmd([sys.executable, "-m", "pip", "install", "--no-cache-dir", cm_dir, "--no-build-isolation"])

    fg_dir = os.path.join(EXT_DIR, "FlexGEMM")
    if os.path.exists(fg_dir): shutil.rmtree(fg_dir)
    run_cmd(["git", "clone", "https://github.com/JeffreyXiang/FlexGEMM.git", fg_dir, "--recursive"])
    run_cmd([sys.executable, "-m", "pip", "install", "--no-cache-dir", fg_dir, "--no-build-isolation"])

t2_dir = os.path.join(SYSTEM_DIRS['models'], "TRELLIS.2")
if FORCE_RECOMPILE or not os.path.exists(os.path.join(t2_dir, "setup.sh")):
    if os.path.exists(t2_dir): shutil.rmtree(t2_dir)
    run_cmd(["git", "clone", "-b", "main", "https://github.com/microsoft/TRELLIS.2.git", t2_dir, "--recursive"])

if FORCE_RECOMPILE or not is_installed("o_voxel"):
    run_cmd([sys.executable, "-m", "pip", "install", "--no-cache-dir", os.path.join(t2_dir, "o-voxel"), "--no-build-isolation"])

sys.path.insert(0, t2_dir)
print("Environment Ready! TRELLIS.2 mapped and ready.")

0. ☢️ Purging stray CUDA 13 binaries and cleaning C++ packages...


1. Enforcing PyTorch 2.6.0 (CUDA 12.4) and compatible xformers...
Looking in indexes: https://download.pytorch.org/whl/cu124
  Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparselt_cu12-0.6.2-py3-none-manylinux2014_x86_64.whl.metadata (6.8 kB)
  Using cached nvidia_nccl_cu12-2.21.5-py3-none-manylinux2014_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cublas_cu12-12.9.2.10-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.9.86-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl.metadata (1.7 kB)
Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl (664.8 MB)
Using cached nvidia_cusparselt_cu12-0.6.2-py3-none-manylinux2014_x86_64.whl (150.1 MB)
Using cached nvidia_nccl_cu12-2.21.5-py3-none-manylinux2014_x86_64.whl (188.7 MB)
Using cached nvidia_cublas_cu12-12.9.2.10-py3-none-manylinux_2_27_x86_64.whl (581.2 MB)
Using cached nvidia_cuda_nvrtc_cu12

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.9.2.10 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-nvrtc-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-nvrtc-cu12 12.9.86 which is incompatible.
xformers 0.0.29.post1 requires torch==2.5.1, but you have torch 2.6.0+cu124 which is incompatible.


2. Installing Basic Dependencies...
  Using cached nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cublas_cu12-12.4.5.8-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
Using cached nvidia_cublas_cu12-12.4.5.8-py3-none-manylinux2014_x86_64.whl (363.4 MB)
Using cached nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl (24.6 MB)
  Attempting uninstall: nvidia-cuda-nvrtc-cu12
    Found existing installation: nvidia-cuda-nvrtc-cu12 12.9.86
    Uninstalling nvidia-cuda-nvrtc-cu12-12.9.86:
      Successfully uninstalled nvidia-cuda-nvrtc-cu12-12.9.86
  Attempting uninstall: nvidia-cublas-cu12━━━━━━ 0/2 [nvidia-cuda-nvrtc-cu12]
    Found existing installation: nvidia-cublas-cu12 12.9.2.10 [nvidia-cuda-nvrtc-cu12]
    Uninstalling nvidia-cublas-cu12-12.9.2.10:0m 0/2 [nvidia-cuda-nvrtc-cu12]
      Successfully uninstalled nvidia-cublas-cu12-12.9.2.100m [nvidia-cuda-nvrtc-cu12]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xformers 0.0.29.post1 requires torch==2.5.1, but you have torch 2.6.0+cu124 which is incompatible.


  Cloning https://github.com/EasternJournalist/utils3d.git (to revision 9a4eb15e4021b67b12c460c7057d642626897ec8) to ./aireos_cache/pip-req-build-j0ubbhy7


  Running command git clone --filter=blob:none --quiet https://github.com/EasternJournalist/utils3d.git /nlsasfs/home/lmedge/rahkus/aireos_cache/pip-req-build-j0ubbhy7
  Running command git rev-parse -q --verify 'sha^9a4eb15e4021b67b12c460c7057d642626897ec8'
  Running command git fetch -q https://github.com/EasternJournalist/utils3d.git 9a4eb15e4021b67b12c460c7057d642626897ec8
  Running command git checkout -q 9a4eb15e4021b67b12c460c7057d642626897ec8


  Resolved https://github.com/EasternJournalist/utils3d.git to commit 9a4eb15e4021b67b12c460c7057d642626897ec8
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
3. ✅ 3D C++ Extensions already detected! Skipping heavy compilation.
Environment Ready! TRELLIS.2 mapped and ready.


In [11]:
import core_config
from core_config import SYSTEM_DIRS

import os
TRELLIS_REPO_ID = "microsoft/TRELLIS.2-4B"
TRELLIS_HF_CACHE_ROOT = os.path.join(
    SYSTEM_DIRS["models"], "hub", "models--microsoft--TRELLIS.2-4B"
)
TRELLIS_SNAPSHOTS_DIR = os.path.join(TRELLIS_HF_CACHE_ROOT, "snapshots")
TRELLIS_LOCAL_CACHE_AVAILABLE = (
    os.path.isdir(TRELLIS_SNAPSHOTS_DIR) and bool(os.listdir(TRELLIS_SNAPSHOTS_DIR))
)
# Set offline mode before importing Gradio/TRELLIS/Hugging Face Hub.
if TRELLIS_LOCAL_CACHE_AVAILABLE:
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"

import sys
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "0"
sys.stdout = sys.__stdout__

import torch
import random
import gc
import traceback
from PIL import Image, ImageOps
import gradio as gr

os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'
os.environ["TRELLIS_CACHE_DIR"] = os.path.join(SYSTEM_DIRS['models'], "trellis_weights")
ATTN_BACKEND_REQUESTED = os.environ.get("AIEROS_TRELLIS_ATTN_BACKEND", "xformers")
if ATTN_BACKEND_REQUESTED not in ("xformers", "flash_attn"):
    raise ValueError(
        f"Unsupported TRELLIS sparse-attention backend {ATTN_BACKEND_REQUESTED!r}; "
        "use xformers or flash_attn (SDPA is not supported by TRELLIS sparse attention)."
    )
os.environ["ATTN_BACKEND"] = ATTN_BACKEND_REQUESTED
os.environ["SPARSE_ATTN_BACKEND"] = ATTN_BACKEND_REQUESTED

# Probe the actual CUDA implementation before importing TRELLIS, so a mismatched
# wheel fails here with an actionable message rather than halfway through generation.
if ATTN_BACKEND_REQUESTED == "xformers":
    try:
        import xformers.ops as xops
        probe = torch.zeros((1, 2, 1, 64), device="cuda", dtype=torch.float16)
        xops.memory_efficient_attention(probe, probe, probe)
        del probe
        print("[ATTENTION] xFormers CUDA operator probe passed.")
    except Exception as exc:
        raise RuntimeError(
            "The installed xFormers CUDA operator is incompatible with this PyTorch/CUDA build. "
            "Run the TRELLIS setup cell in phase2_3d to install the pinned compatible wheel, "
            "or set AIEROS_TRELLIS_ATTN_BACKEND=flash_attn if flash_attn is installed."
        ) from exc
else:
    import flash_attn
    print("[ATTENTION] Using flash_attn for dense and sparse attention.")

from trellis2.pipelines import Trellis2ImageTo3DPipeline
import o_voxel
from storage_manager import managed_storage_operation
from storage_policy import BYTES_PER_GB, estimate_hf_repo_bytes
from purge_manager import DiskPurgeManager

OUTPUT_DIR = os.path.join(SYSTEM_DIRS['outputs'], "3d_models")
os.makedirs(OUTPUT_DIR, exist_ok=True)

def load_trellis_pipeline():
    repo_id = TRELLIS_REPO_ID
    if TRELLIS_LOCAL_CACHE_AVAILABLE:
        print("TRELLIS.2-4B cache detected; loading strictly offline.")
        try:
            pipe = Trellis2ImageTo3DPipeline.from_pretrained(repo_id)
            print("TRELLIS cache hit; no model download required.")
            return pipe
        except Exception as exc:
            raise RuntimeError(
                "A TRELLIS cache snapshot exists but offline loading failed. "
                "Automatic re-download is disabled to avoid duplicating a large checkpoint; "
                "inspect the cached snapshot and the underlying traceback."
            ) from exc

    required_gb = (
        estimate_hf_repo_bytes(repo_id) / BYTES_PER_GB
        + max(0.0, float(os.environ.get("AIEROS_MODEL_DOWNLOAD_SAFETY_GB", "2.0")))
    )
    with managed_storage_operation(
        "models",
        required_gb=required_gb,
        require_fresh_quota=True,
        minimum_free_gb=float(os.environ.get("AIEROS_MODEL_MIN_FREE_GB", "2.0")),
    ):
        return Trellis2ImageTo3DPipeline.from_pretrained(repo_id)

trellis_pipe = load_trellis_pipeline()
trellis_pipe.cuda()

if ENABLE_DISK_PURGE and os.environ.get("AIEROS_ALLOW_SHARED_MODEL_PURGE") == "1":
    print("\n[WARNING] Explicit shared-cache purge requested. This is unsafe if any other process may still be loading the same model.")
    DiskPurgeManager.purge_hf_model("microsoft/TRELLIS.2-4B")
else:
    print("[STORAGE] Shared model/cache files retained; no automatic shared-cache purge.")
# TRELLIS_CACHE_DIR is retained because runtime components may use it lazily.

@torch.inference_mode()
def extrude_3d_asset(input_image, decimation_target: int):
    if input_image is None: 
        raise gr.Error("Please upload an image first!")
        
    seed = random.randint(0, 999999)
    print(f"\n[EXTRUDING] Processing uploaded image | Seed: {seed}")
    
    try:
        input_image = input_image.convert("RGB")
        w, h = input_image.size
        max_size = 1024
        
        if max(w, h) > max_size:
            scale = max_size / float(max(w, h))
            w = int(w * scale)
            h = int(h * scale)
            input_image = input_image.resize((w, h), Image.Resampling.LANCZOS)
            
        sq_size = max(w, h)
        sq_size = max(64, int(round(sq_size / 64.0)) * 64)
        
        square_img = Image.new("RGB", (sq_size, sq_size), (255, 255, 255))
        square_img.paste(input_image, ((sq_size - w) // 2, (sq_size - h) // 2))
        input_image = square_img
            
        torch.cuda.empty_cache()
            
        mesh = trellis_pipe.run(input_image)[0]
        mesh.simplify(16777216) 
        
        glb_path = os.path.join(OUTPUT_DIR, f"standalone_t2_{seed}.glb")
        glb = o_voxel.postprocess.to_glb(
            vertices            =   mesh.vertices,
            faces               =   mesh.faces,
            attr_volume         =   mesh.attrs,
            coords              =   mesh.coords,
            attr_layout         =   mesh.layout,
            voxel_size          =   mesh.voxel_size,
            aabb                =   [[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],
            decimation_target   =   int(decimation_target),
            texture_size        =   4096,
            remesh              =   True,
            remesh_band         =   1,
            remesh_project      =   0,
            verbose             =   False
        )
        with managed_storage_operation(
            "outputs",
            required_gb=0.20,
            require_fresh_quota=False,
            minimum_free_gb=0.05,
        ):
            glb.export(glb_path, extension_webp=True)
        
        del mesh, glb
        gc.collect()
        torch.cuda.empty_cache()
        return glb_path
        
    except Exception as e:
        error_trace = traceback.format_exc()
        print(f"\n[CRITICAL ERROR]\n{error_trace}")
        gc.collect()
        torch.cuda.empty_cache()
        raise gr.Error(f"Generation Failed: {str(e)}")

with gr.Blocks(title="AIEROS TRELLIS.2 Standalone") as demo:
    gr.Markdown("# 🕹️ AIEROS Studio: TRELLIS.2-4B (Isolated Testing)")
    with gr.Row():
        with gr.Column(scale=1):
            input_image = gr.Image(label="Upload 2D Concept", type="pil")
            decimation = gr.Slider(label="Target Polygon Count", minimum=10000, maximum=500000, value=100000, step=10000)
            generate_btn = gr.Button("Extrude to 3D", variant="primary", size="lg")
        with gr.Column(scale=2):
            output_3d = gr.Model3D(label="TRELLIS.2 3D Viewer", clear_color=[0.1, 0.1, 0.1, 1.0])

    generate_btn.click(fn=extrude_3d_asset, inputs=[input_image, decimation], outputs=[output_3d])

print("Starting Gradio Web Server on port 7862...")
demo.launch(server_name="0.0.0.0", server_port=7862, inline=True, share=False, show_error=True)


[SPARSE] Conv backend: flex_gemm; Attention backend: xformers


2026-09-13 00:25:03,134 - [STORAGE] Storage lock RELEASED on 'models'. Download may proceed safely.


Loading TRELLIS.2-4B...
[ATTENTION] Using backend: xformers


/nlsasfs/home/lmedge/rahkus/Conda/envs/phase2_3d/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/nlsasfs/home/lmedge/rahkus/Conda/envs/phase2_3d/lib/python3.10/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
2026-09-13 00:28:47,528 - [STORAGE] Storage lock RESUMED on 'models'.


Starting Gradio Web Server on port 7862...


2026-09-13 00:28:48,846 - [STORAGE] HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
2026-09-13 00:28:48,851 - [STORAGE] HTTP Request: GET http://localhost:7862/gradio_api/startup-events "HTTP/1.1 200 OK"
2026-09-13 00:28:48,917 - [STORAGE] HTTP Request: HEAD http://localhost:7862/ "HTTP/1.1 200 OK"


* Running on local URL:  http://0.0.0.0:7862
* To create a public link, set `share=True` in `launch()`.
Sun Sep 13 00:28:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.07              Driver Version: 550.90.07      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          On  |   00000000:07:00.0 Off |                    0 |
| N/A   31C    P0             50W /  400W |       1MiB /  40960MiB |      0%      Default |
|                                   

In [12]:
# demo.close()

In [13]:
!nvidia-smi